# OMI → Colab → Google Drive

Archive original FITS, screen each exposure, register and integrate suitable lights on the GPU, and save linear masters plus finished PNG/TIFF presentations to Drive.

**Next session:** paste the shared Dropbox link for that observing-night folder, choose **New night folder**, enter its date label and the target name shown in the FITS header, then **Run all**. Your laptop does not download or process the images. A personal Dropbox account is not required.

Use **Runtime → Change runtime type → L4 GPU** (another CUDA GPU also works). Google may ask you to mount Drive. Keep this notebook private if you enter access-bearing shared links.

The initial archive can be resumed with **Saved manifest**. **Already archived** skips transfer. This is a run-on-demand notebook, not a permanent Dropbox watcher; Colab sessions have time and availability limits.

In [ ]:
#@title 1. Choose source and processing target
IMPORT_MODE = "Saved manifest" #@param ["Saved manifest", "New night folder", "Already archived"]
NEW_NIGHT_LINK = "" #@param {type:"string"}
NEW_NIGHT_LABEL = "" #@param {type:"string"}
TARGET = "Heart Nebula" #@param {type:"string"}
OBSERVING_NIGHT = "2026-11-01" #@param {type:"string"}
REFERENCE_PATH = "" #@param {type:"string"}
BINNING = 2 #@param [1, 2, 4] {type:"raw"}
RUN_PROCESSING = True #@param {type:"boolean"}
FINISH_IMAGES = True #@param {type:"boolean"}
DISPLAY_IMAGES = False #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/Astro/OMI_Archive" #@param {type:"string"}


In [ ]:
#@title 2. Connect Drive and install the pinned Alcor core
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, subprocess, json, hashlib, re
import importlib.metadata as metadata
ROOT=Path(ARCHIVE_PATH); ROOT.mkdir(parents=True,exist_ok=True)
WORK=Path('/content/omi_workflow'); WORK.mkdir(exist_ok=True)
subprocess.run([sys.executable,'-m','pip','-q','install','sep','astroalign','pytest',
               'git+https://github.com/AltairSo/alcor-imaging.git@23bbf079057d1546f55a0c158f74f4d38cd7ee05'],check=True)
driver_report=subprocess.check_output(['nvidia-smi'],text=True)
cuda_major=int(re.search(r'CUDA Version:\s*(\d+)',driver_report).group(1))
cupy_package='cupy-cuda13x' if cuda_major>=13 else 'cupy-cuda12x'
installed=[d.metadata['Name'] for d in metadata.distributions() if d.metadata['Name'] in ['cupy','cupy-cuda11x','cupy-cuda12x','cupy-cuda13x']]
if installed!=[cupy_package]:
    if installed:subprocess.run([sys.executable,'-m','pip','-q','uninstall','-y',*installed],check=True)
    subprocess.run([sys.executable,'-m','pip','-q','install','--no-deps',cupy_package],check=True)
sys.path.insert(0,str(WORK))
from alcor_imaging import gpu_available
assert gpu_available(), 'Select a GPU runtime before processing.'
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True))


In [ ]:
#@title 3. Load the versioned workflow (included in this notebook)
MODULES = {'archive.py': '"""Dropbox shared-link intake; runs in Colab, never needs a Dropbox secret.\n\nKeep source manifests private: shared URLs are access-bearing links.\nThe Alcor core remains independent of storage providers.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport shutil\nimport time\nimport zipfile\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nfrom pathlib import Path, PurePosixPath\nfrom urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit\n\nimport requests\n\n\ndef safe_relative(value):\n    p = PurePosixPath(value)\n    if p.is_absolute() or not p.parts or any(x in {"..", "."} for x in p.parts) or "\\\\" in value:\n        raise ValueError("Unsafe archive path")\n    return Path(*p.parts)\n\n\ndef download_url(url):\n    p = urlsplit(url)\n    if p.scheme != "https" or p.hostname not in {"dropbox.com", "www.dropbox.com", "dl.dropboxusercontent.com"}:\n        raise ValueError("Expected an HTTPS Dropbox shared link")\n    query = dict(parse_qsl(p.query)); query.pop("raw", None); query["dl"] = "1"\n    return urlunsplit((p.scheme, p.netloc, p.path, urlencode(query), ""))\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for chunk in iter(lambda: f.read(8 * 1024**2), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef atomic_json(path, data):\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + ".part")\n    tmp.write_text(json.dumps(data, indent=2, allow_nan=False, default=str))\n    os.replace(tmp, path)\n\n\ndef validate_fits(path):\n    from astropy.io import fits\n    with fits.open(path, memmap=False, lazy_load_hdus=False) as hdus:\n        hdus.verify("exception")\n        images = [h for h in hdus if int(h.header.get("NAXIS", 0)) >= 2]\n        if not images:\n            raise ValueError("FITS has no image HDU")\n        for h in images:\n            info = h.fileinfo()\n            if info and info["datLoc"] + info["datSpan"] > Path(path).stat().st_size:\n                raise ValueError("Truncated FITS payload")\n            if "CHECKSUM" in h.header and h.verify_checksum() != 1:\n                raise ValueError("FITS CHECKSUM mismatch")\n\n\ndef import_one(item, root, retries=4):\n    root = Path(root)\n    relative = safe_relative(item["path"])\n    dest = root / "raw" / relative\n    receipt = root / "receipts" / relative.with_name(relative.name + ".json")\n    if dest.exists() and receipt.exists():\n        old = json.loads(receipt.read_text())\n        if old.get("bytes") == dest.stat().st_size and old.get("sha256") == digest(dest):\n            return {**old, "status": "verified_existing"}\n    # A runtime may stop between moving the raw file and writing its receipt.\n    # Re-download to a temporary file, compare, and adopt only identical bytes.\n    dest.parent.mkdir(parents=True, exist_ok=True)\n    tmp = dest.with_name(dest.name + ".part")\n    last = None\n    for attempt in range(retries):\n        try:\n            # Restart this one incomplete file, while retaining all completed files.\n            h = hashlib.sha256(); size = 0\n            with requests.get(download_url(item["url"]), stream=True, timeout=(30, 180)) as r:\n                r.raise_for_status()\n                if "text/html" in r.headers.get("Content-Type", ""):\n                    raise ValueError("Dropbox returned a web page rather than a file")\n                expected = int(r.headers.get("Content-Length", 0))\n                with tmp.open("wb") as f:\n                    for chunk in r.iter_content(4 * 1024**2):\n                        if chunk:\n                            f.write(chunk); h.update(chunk); size += len(chunk)\n                if expected and size != expected:\n                    raise ValueError("Incomplete HTTP payload")\n            fits_error = None\n            if relative.suffix.lower() in {".fits", ".fit", ".fts"}:\n                try:\n                    validate_fits(tmp)\n                except Exception as e:\n                    # Preserve a damaged source too; quality screening excludes it.\n                    fits_error = type(e).__name__ + ": " + str(e)\n            if dest.exists():\n                if digest(dest) != h.hexdigest():\n                    raise FileExistsError(f"Existing original differs: {relative}")\n                tmp.unlink()\n            else:\n                os.replace(tmp, dest)\n            result = {"path": str(relative), "bytes": size, "sha256": h.hexdigest(),\n                      "night": item.get("night", ""), "status": "downloaded", "fits_error": fits_error,\n                      "archived_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n            atomic_json(receipt, result)\n            return result\n        except Exception as e:\n            last = type(e).__name__ + ": " + str(e).split(" for url:")[0]\n            if attempt + 1 < retries:\n                time.sleep(min(30, 2 ** (attempt + 1)))\n    # Do not log access-bearing URLs.\n    return {"path": str(relative), "status": "failed", "error": last}\n\n\ndef import_manifest(manifest, root, workers=4):\n    root = Path(root); root.mkdir(parents=True, exist_ok=True)\n    files = manifest["files"]\n    paths = [str(safe_relative(x["path"])) for x in files]\n    if len(paths) != len(set(paths)):\n        raise ValueError("Duplicate destination paths in manifest")\n    atomic_json(root / "source_manifest.private.json", manifest)\n    results = []\n    with ThreadPoolExecutor(max_workers=workers) as pool:\n        jobs = {pool.submit(import_one, item, root): item for item in files}\n        for job in as_completed(jobs):\n            try:\n                r = job.result()\n            except Exception as e:\n                r = {"path": jobs[job]["path"], "status": "failed", "error": str(e)}\n            results.append(r)\n            print(f"{len(results)}/{len(files)} {r[\'status\']}: {r[\'path\']}", flush=True)\n            atomic_json(root / "transfer_report.json", results)\n    return results\n\n\ndef import_folder_link(url, root, label, staging="/content/omi_staging", *, refresh=True, keep_staging=False):\n    """Future runs: paste one observing-night folder link, no file list needed.\n\n    Downloads a ZIP to Colab\'s temporary disk, safely extracts into Drive,\n    checks FITS and hashes, and keeps completed originals on repeat runs.\n    Large archives should be split into observing-night links to fit disk.\n    """\n    relative = safe_relative(label); root = Path(root)\n    stage = Path(staging); stage.mkdir(parents=True, exist_ok=True)\n    key = hashlib.sha256(url.encode()).hexdigest()[:16]\n    zip_path = stage / (key + ".zip")\n    # A shared night folder may receive more exposures. Do not silently reuse\n    # an older ZIP snapshot; reuse is only an explicit extraction-resume option.\n    if refresh or not zip_path.exists() or not zipfile.is_zipfile(zip_path):\n        tmp = zip_path.with_suffix(".part")\n        with requests.get(download_url(url), stream=True, timeout=(30, 180)) as r:\n            r.raise_for_status()\n            if "text/html" in r.headers.get("Content-Type", ""):\n                raise ValueError("Shared link did not return a downloadable folder")\n            n = int(r.headers.get("Content-Length", 0))\n            if n and n > shutil.disk_usage(stage).free - 1024**3:\n                raise ValueError("Not enough Colab disk; use a smaller night folder")\n            with tmp.open("wb") as out:\n                for chunk in r.iter_content(8 * 1024**2):\n                    if shutil.disk_usage(stage).free < 512 * 1024**2:\n                        raise ValueError("Colab staging disk is full; use a smaller folder")\n                    out.write(chunk)\n        if not zipfile.is_zipfile(tmp):\n            raise ValueError("Dropbox response is not a ZIP archive")\n        os.replace(tmp, zip_path)\n    results = []\n    with zipfile.ZipFile(zip_path) as archive:\n        for info in archive.infolist():\n            if info.is_dir():\n                continue\n            member = safe_relative(info.filename)\n            if (info.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError("ZIP symlinks are not accepted")\n            rel = relative / member\n            dest = root / "raw" / rel\n            receipt = root / "receipts" / rel.with_name(rel.name + ".json")\n            dest.parent.mkdir(parents=True, exist_ok=True)\n            tmp = dest.with_name(dest.name + ".part")\n            h = hashlib.sha256()\n            with archive.open(info) as src, tmp.open("wb") as out:\n                for chunk in iter(lambda: src.read(8 * 1024**2), b""):\n                    out.write(chunk); h.update(chunk)\n            fits_error = None\n            if dest.suffix.lower() in {".fit", ".fits", ".fts"}:\n                try:\n                    validate_fits(tmp)\n                except Exception as e:\n                    fits_error = type(e).__name__ + ": " + str(e)\n            sha = h.hexdigest()\n            if dest.exists():\n                if digest(dest) != sha:\n                    raise FileExistsError(f"Original differs; choose another label: {rel}")\n                tmp.unlink()  # only the newly generated temporary copy\n            else:\n                os.replace(tmp, dest)\n            r = {"path": str(rel), "sha256": sha, "bytes": info.file_size,\n                 "night": label, "status": "archived", "fits_error": fits_error}\n            atomic_json(receipt, r); results.append(r)\n    atomic_json(root / "folder_imports" / (key + ".json"), results)\n    if not keep_staging:\n        zip_path.unlink()  # verified originals and receipts are already on Drive\n    return results\n', 'quality.py': '"""Conservative FITS screening and review sheets, run on cloud compute.\n\nNo frame is discarded. Thresholds are screening heuristics, not a claim that\nMoon illumination alone establishes image usability. Elongated detections are\nincluded when measuring tracking: filtering them first hides failed guiding.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport re\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\nfrom pathlib import Path\n\nimport numpy as np\nimport sep\nfrom astropy.io import fits\nfrom PIL import Image, ImageDraw\nfrom scipy.ndimage import gaussian_filter1d, median_filter\n\nfrom archive import atomic_json, validate_fits\n\nQC_VERSION = 4\n\n\ndef bin_mean(a, factor=4):\n    h, w = a.shape\n    return a[:h//factor*factor, :w//factor*factor].reshape(\n        h//factor, factor, w//factor, factor).mean((1, 3), dtype=np.float32)\n\n\ndef mad(a):\n    a = np.asarray(a)\n    return float(1.4826 * np.nanmedian(np.abs(a - np.nanmedian(a))))\n\n\ndef display_mono(a, size=None):\n    a = np.nan_to_num(a.astype(np.float32))\n    lo, hi = np.percentile(a, [10, 99.7])\n    scaled = np.clip((a-lo)/max(hi-lo, 1e-6), 0, None)\n    scaled = np.clip(np.arcsinh(12*scaled)/np.arcsinh(12), 0, 1)\n    im = Image.fromarray(np.round(scaled*255).astype(\'uint8\'))\n    if size:\n        im.thumbnail(size, Image.Resampling.LANCZOS)\n    return im\n\n\ndef star_measurements(data):\n    """Native-pixel samples across the sensor; no elongation cut here."""\n    h, w = data.shape; rows = []; crops = []; noise = []\n    for fy, fx in [(0.5,0.5),(0.22,0.22),(0.22,0.78),(0.78,0.22),(0.78,0.78)]:\n        y = max(0, min(h-512, int(h*fy)-256)); x = max(0, min(w-512, int(w*fx)-256))\n        cut = np.ascontiguousarray(data[y:y+512,x:x+512],dtype=np.float32)\n        noise.append(mad(cut[:,1:]-cut[:,:-1])/np.sqrt(2))\n        bg = sep.Background(cut, bw=64,bh=64)\n        obj = sep.extract(cut-bg.back(), 7, err=bg.rms(), minarea=6)\n        obj = obj[(obj[\'a\']>1.1)&(obj[\'b\']>0.8)&(obj[\'npix\']<700)&\n                  (obj[\'flux\']/np.maximum(obj[\'peak\'],1)>5)&\n                  (obj[\'peak\']+float(bg.globalback)<60000)]\n        rows.extend([(float(o[\'a\']),float(o[\'b\']),float(o[\'theta\'])) for o in obj])\n        if len(obj):\n            o = obj[np.argmax(obj[\'flux\'])]; sx,sy=int(o[\'x\']),int(o[\'y\'])\n            sx=max(40,min(cut.shape[1]-40,sx));sy=max(40,min(cut.shape[0]-40,sy))\n            crops.append(cut[sy-40:sy+40,sx-40:sx+40])\n        else:\n            crops.append(cut[216:296,216:296])\n    if rows:\n        a,b,theta=np.asarray(rows).T; ratio=b/a\n        summary={"sample_stars":len(rows),"axis_ratio":float(np.median(ratio)),\n                 "trail_fraction":float(np.mean(ratio<0.5)),\n                 "fwhm_px":float(np.median(2.355*np.sqrt(a*b))),\n                 "trail_coherence":float(abs(np.mean(np.exp(2j*theta))))}\n    else:\n        summary={"sample_stars":0,"axis_ratio":None,"trail_fraction":None,\n                 "fwhm_px":None,"trail_coherence":None}\n    summary[\'noise_adu\']=float(np.median(noise))\n    return summary,crops\n\n\ndef refine_field_metrics(result,small):\n    """Wide-field morphology catches long arcs missed by sparse native crops."""\n    a=np.ascontiguousarray(small,dtype=np.float32)\n    bg=sep.Background(a,bw=64,bh=64)\n    obj=sep.extract(a-bg.back(),5,err=bg.rms(),minarea=5)\n    obj=obj[(obj[\'a\']>0.65)&(obj[\'b\']>0.45)&(obj[\'npix\']<2000)&\n            (obj[\'flux\']/np.maximum(obj[\'peak\'],1)>3)]\n    result[\'field_stars\']=len(obj)\n    if len(obj):\n        ratio=obj[\'b\']/obj[\'a\']\n        result[\'field_axis_ratio\']=float(np.median(ratio))\n        result[\'field_trail_fraction\']=float(np.mean(ratio<0.5))\n        result[\'field_fwhm_native_px\']=float(np.median(2.355*np.sqrt(obj[\'a\']*obj[\'b\']))*4)\n        if result[\'status\']!=\'planetary_or_test\' and len(obj)>=20:\n            if np.median(ratio)<0.45 and np.mean(ratio<0.5)>0.65:\n                result[\'status\']=\'reject\'\n                result[\'reasons\'].append(\'severe field-wide stellar trails/arcs\')\n            elif np.median(ratio)<0.65 and result[\'status\']==\'candidate\':\n                result[\'status\']=\'review\';result[\'reasons\'].append(\'field-wide stellar elongation\')\n            elif (len(obj)>=30 and np.median(ratio)>=0.75 and\n                  result[\'reasons\']==[\'too few stellar detections for reliable tracking assessment\']):\n                # Sparse native crops need not veto a well-measured full field.\n                result[\'status\']=\'candidate\';result[\'reasons\']=[]\n    result[\'row_jump_noise_ratio\']=float(result.get(\'row_jump_adu\',0))/max(float(result.get(\'noise_adu\',1)),1e-6)\n    if result[\'row_jump_noise_ratio\']>3 and result[\'status\'] not in {\'reject\',\'planetary_or_test\'}:\n        result[\'status\']=\'review\';result[\'reasons\'].append(\'abrupt full-width row discontinuity\')\n    result[\'reasons\']=list(dict.fromkeys(result[\'reasons\']))\n    result[\'qc_version\']=QC_VERSION\n    return result\n\n\ndef upgrade_cached(result,root):\n    root=Path(root)\n    cache=root/\'quality\'/\'cache\'/(result[\'id\']+\'.npy\')\n    if not cache.exists():return inspect_one(root/\'raw\'/result[\'path\'],root)\n    result=refine_field_metrics(result,np.load(cache))\n    atomic_json(root/\'quality\'/\'frames\'/(result[\'id\']+\'.json\'),result)\n    return result\n\n\ndef inspect_one(path, root):\n    root=Path(root);path=Path(path);rel=path.relative_to(root/\'raw\')\n    key=hashlib.sha256(str(rel).encode()).hexdigest()[:20]\n    out=root/\'quality\'/\'frames\';out.mkdir(parents=True,exist_ok=True)\n    result={"path":str(rel),"id":key,"qc_version":QC_VERSION,\n            "night":rel.parts[0] if re.fullmatch(r\'\\d{4}-\\d{2}-\\d{2}\',rel.parts[0]) else \'root\'}\n    try:\n        validate_fits(path)\n        with fits.open(path,memmap=False) as hdus:\n            hdu=next(h for h in hdus if int(h.header.get(\'NAXIS\',0))==2)\n            header=hdu.header.copy(); data=np.asarray(hdu.data,dtype=np.float32)\n        for keyname,kw in [(\'object\',\'OBJECT\'),(\'filter\',\'FILTER\'),(\'exposure\',\'EXPTIME\'),\n                           (\'date_obs\',\'DATE-OBS\'),(\'camera\',\'INSTRUME\'),(\'gain\',\'GAIN\'),\n                           (\'offset\',\'OFFSET\'),(\'temperature\',\'CCD-TEMP\'),(\'bin_x\',\'XBINNING\'),\n                           (\'bin_y\',\'YBINNING\'),(\'ra\',\'RA\'),(\'dec\',\'DEC\'),(\'image_type\',\'IMAGETYP\')]:\n            result[keyname]=header.get(kw)\n        result[\'shape\']=list(data.shape)\n        finite=np.isfinite(data);result[\'nonfinite_fraction\']=float(1-np.mean(finite))\n        if result[\'nonfinite_fraction\']>0.01 or not np.any(finite):\n            raise ValueError(\'Invalid/nonfinite pixel payload\')\n        data=np.nan_to_num(data,nan=float(np.nanmedian(data)))\n        result[\'saturated_fraction\']=float(np.mean(data>=65000))\n        small=bin_mean(data,4)\n        bg=sep.Background(np.ascontiguousarray(small),bw=64,bh=64)\n        result[\'background_adu\']=float(bg.globalback)\n        result[\'gradient_adu\']=float(np.percentile(bg.back(),95)-np.percentile(bg.back(),5))\n        stars,crops=star_measurements(data);result.update(stars)\n        # Detect narrow row/column offsets without fitting away nebular structure.\n        row=np.median(data[:,::8],axis=1);col=np.median(data[::8,:],axis=0)\n        row_res=row-median_filter(row,size=15,mode=\'nearest\')\n        col_res=col-median_filter(col,size=15,mode=\'nearest\')\n        result[\'band_amplitude_adu\']=float(max(np.percentile(abs(row_res),95),np.percentile(abs(col_res),95)))\n        result[\'band_noise_ratio\']=result[\'band_amplitude_adu\']/max(result[\'noise_adu\'],1e-6)\n        result[\'row_jump_adu\']=float(np.max(abs(np.diff(gaussian_filter1d(row,2)))))\n        result[\'signal_span_adu\']=float(np.percentile(small,99)-np.percentile(small,20))\n        result[\'status\']=\'candidate\'; reasons=[]\n        if result[\'signal_span_adu\']<=0 or result[\'noise_adu\']<=0:\n            result[\'status\']=\'reject\';reasons.append(\'constant or degenerate image\')\n        if result[\'sample_stars\']>=12 and result[\'axis_ratio\']<0.40 and result[\'trail_fraction\']>0.7:\n            result[\'status\']=\'reject\';reasons.append(\'severe elongated stellar profiles\')\n        elif result[\'sample_stars\']>=8 and result[\'axis_ratio\']<0.65:\n            result[\'status\']=\'review\';reasons.append(\'elongated stellar profiles\')\n        if result[\'band_noise_ratio\']>0.7:\n            if result[\'status\']!=\'reject\':result[\'status\']=\'review\'\n            reasons.append(\'row/column pattern needs visual review\')\n        if result[\'sample_stars\']<8:\n            if result[\'status\']!=\'reject\':result[\'status\']=\'review\'\n            reasons.append(\'too few stellar detections for reliable tracking assessment\')\n        if float(result.get(\'exposure\') or 0)<1 or \'SNAPSHOT\' in rel.parts:\n            result[\'status\']=\'planetary_or_test\';reasons.append(\'separate planetary/test workflow\')\n        result[\'reasons\']=reasons\n        result=refine_field_metrics(result,small)\n        sheet=Image.new(\'RGB\',(960,760),(18,20,25));draw=ImageDraw.Draw(sheet)\n        thumb=display_mono(small,(940,570));sheet.paste(thumb,((960-thumb.width)//2,40))\n        draw.text((12,8),f"{result.get(\'object\')} | {result.get(\'filter\')} | {result.get(\'exposure\')} s | {result[\'night\']}",fill=\'white\')\n        for i,crop in enumerate(crops):\n            sheet.paste(display_mono(crop).resize((150,150),Image.Resampling.NEAREST),(10+i*190,600))\n        draw.text((12,575),f"{result[\'status\']}   axis b/a={result[\'axis_ratio\']}   FWHM={result[\'fwhm_px\']} px   native star crops below",fill=\'white\')\n        sheet.save(out/(key+\'.jpg\'),quality=88)\n        # Small linear data and catalog support quick, deterministic review stacks.\n        cache=root/\'quality\'/\'cache\';cache.mkdir(parents=True,exist_ok=True)\n        np.save(cache/(key+\'.npy\'),small)\n    except Exception as e:\n        status=\'non_image\' if str(e)==\'FITS has no image HDU\' else \'reject\'\n        result.update(status=status,reasons=[type(e).__name__+\': \'+str(e)])\n    atomic_json(out/(key+\'.json\'),result)\n    return result\n\n\ndef scan(root, workers=4):\n    root=Path(root);results=[]\n    paths=sorted(p for p in (root/\'raw\').rglob(\'*\') if p.suffix.lower() in {\'.fits\',\'.fit\',\'.fts\'})\n    report=root/\'quality\'/\'frame_report.json\'\n    previous={r[\'path\']:r for r in json.loads(report.read_text())} if report.exists() else {}\n    pending=[];upgrades=[]\n    for i,p in enumerate(paths):\n        rel=p.relative_to(root/\'raw\');key=hashlib.sha256(str(rel).encode()).hexdigest()[:20]\n        saved=root/\'quality\'/\'frames\'/(key+\'.json\')\n        old=previous.get(str(rel))\n        if old is None and saved.exists():old=json.loads(saved.read_text())\n        if old and old.get(\'qc_version\')==QC_VERSION:\n            results.append(old)\n        elif old and old.get(\'qc_version\') in {2,3}:\n            upgrades.append(old)\n        else:\n            pending.append(p)\n    with ProcessPoolExecutor(max_workers=workers) as pool:\n        jobs=[pool.submit(upgrade_cached,r,root) for r in upgrades]+[pool.submit(inspect_one,p,root) for p in pending]\n        for job in as_completed(jobs):\n            result=job.result();results.append(result)\n            print(f"QC {len(results)}/{len(paths)} {result[\'status\']}: {result[\'path\']}",flush=True)\n            if len(results)%25==0:\n                atomic_json(\'/content/omi_qc_live.json\',results)\n                atomic_json(report,results)\n    atomic_json(root/\'quality\'/\'frame_report.json\',results)\n    columns=[\'night\',\'object\',\'filter\',\'exposure\',\'status\',\'sample_stars\',\'fwhm_px\',\'axis_ratio\',\n             \'trail_fraction\',\'background_adu\',\'noise_adu\',\'gradient_adu\',\'band_noise_ratio\',\n             \'saturated_fraction\',\'reasons\',\'path\']\n    with (root/\'quality\'/\'frame_report.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=columns,extrasaction=\'ignore\');writer.writeheader()\n        for r in results:writer.writerow(r)\n    return results\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'root\');args=parser.parse_args()\n    scan(args.root)\n', 'process.py': '"""GPU registration/integration with auditable frame selection and quicklooks.\n\nDefaults to a 4x4 review product. Set binning=1 for native-resolution masters.\nNo outside imagery, generative restoration, or local colour patch mapping.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport re\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport astroalign\nimport numpy as np\nimport sep\nimport tifffile\nfrom astropy.io import fits\nfrom PIL import Image\nfrom scipy.ndimage import gaussian_filter\n\nfrom alcor_imaging import StackConfig, stack, warp_affine\nfrom archive import atomic_json\nfrom quality import bin_mean, display_mono\n\n\ndef catalog(image):\n    a=np.ascontiguousarray(np.nan_to_num(image),dtype=np.float32)\n    bg=sep.Background(a,bw=64,bh=64)\n    obj=sep.extract(a-bg.back(),5,err=bg.rms(),minarea=5)\n    obj=obj[(obj[\'a\']>0)&(obj[\'b\']>0)&(obj[\'b\']/np.maximum(obj[\'a\'],1e-6)>0.45)&\n            (obj[\'npix\']<400)&(obj[\'flux\']/np.maximum(obj[\'peak\'],1)>3)]\n    obj=obj[np.argsort(obj[\'flux\'])[::-1]][:100]\n    return np.column_stack([obj[\'x\'],obj[\'y\']])\n\n\ndef transform(source, reference):\n    if len(source)<12 or len(reference)<12:\n        raise ValueError(\'Fewer than 12 stars for registration\')\n    t,(s,d)=astroalign.find_transform(source,reference,max_control_points=80)\n    rms=float(np.sqrt(np.mean(np.sum((t(s)-d)**2,axis=1))))\n    if not 0.98<t.scale<1.02 or rms>0.8 or len(s)<12:\n        raise ValueError(f\'Registration failed validation: matches={len(s)}, RMS={rms:.3f}, scale={t.scale:.4f}\')\n    return t.params,{\'matches\':len(s),\'rms_binned_px\':rms,\'scale\':float(t.scale)}\n\n\ndef rebin_matrix(matrix, source_bin=4, dest_bin=1):\n    factor=source_bin/dest_bin\n    s=np.array([[factor,0,(factor-1)/2],[0,factor,(factor-1)/2],[0,0,1.]])\n    return s@matrix@np.linalg.inv(s)\n\n\ndef compatible_key(r):\n    return (tuple(r[\'shape\']),r.get(\'camera\'),r.get(\'gain\'),r.get(\'offset\'),r.get(\'bin_x\'),r.get(\'bin_y\'))\n\n\ndef choose_reference(selected, reference_path=None):\n    """Allow an inspected field to anchor registration after a pointing failure.\n\n    Sharp stars alone do not establish that an exposure contains the target.\n    An explicit reference must still be in the accepted, compatible selection.\n    All other exposures must pass the normal stellar registration checks.\n    """\n    if reference_path is not None:\n        matches=[r for r in selected if r[\'path\']==str(reference_path)]\n        if len(matches)!=1:\n            raise ValueError(\'Reference path must identify exactly one selected light frame\')\n        return matches[0]\n    return min(selected,key=lambda r:(r.get(\'filter\') not in {\'Ha\',\'L\'},r.get(\'fwhm_px\') or 100))\n\n\ndef process_target(root,target,night=None,binning=4,backend=\'gpu\',include_review=False,reference_path=None):\n    root=Path(root)\n    rows=json.loads((root/\'quality\'/\'frame_report.json\').read_text())\n    overrides=json.loads((root/\'quality\'/\'overrides.json\').read_text()) if (root/\'quality\'/\'overrides.json\').exists() else {}\n    selected=[]\n    for r in rows:\n        if r.get(\'object\')!=target or (night and r.get(\'night\')!=night):continue\n        status=overrides.get(r[\'path\'],{}).get(\'status\',r[\'status\'])\n        if status not in ({\'candidate\',\'review\',\'accepted\'} if include_review else {\'candidate\',\'accepted\'}):continue\n        if str(r.get(\'image_type\',\'\')).upper() not in {\'LIGHT\',\'LIGHT FRAME\'}:continue\n        if float(r.get(\'exposure\') or 0)<1:continue\n        selected.append(r)\n    if not selected:raise ValueError(\'No accepted light frames for this target/night\')\n    if len({compatible_key(r) for r in selected})!=1:\n        raise ValueError(\'Incompatible detector/gain/geometry groups: process them separately\')\n    if binning not in {1,2,4}:raise ValueError(\'binning must be 1, 2 or 4\')\n    label=re.sub(r\'[^A-Za-z0-9_-]+\',\'_\',target).strip(\'_\')\n    ref=choose_reference(selected,reference_path)\n    run_hash=hashlib.sha256(json.dumps({\'paths\':sorted(r[\'path\'] for r in selected),\'binning\':binning,\n                                      \'reference\':ref[\'path\'],\'version\':2},sort_keys=True).encode()).hexdigest()[:10]\n    out=root/\'processed\'/label/(str(night or \'compatible_nights\')+f\'_bin{binning}_{run_hash}\')\n    out.mkdir(parents=True,exist_ok=True)\n    def cache(r):return np.load(root/\'quality\'/\'cache\'/(r[\'id\']+\'.npy\'))\n    ref_small=cache(ref);ref_cat=catalog(ref_small)\n    grouped=defaultdict(list);registration=[]\n    for r in sorted(selected,key=lambda r:r[\'path\']):\n        try:\n            small=cache(r)\n            matrix,stats=(np.eye(3),{\'matches\':len(ref_cat),\'rms_binned_px\':0.,\'scale\':1.}) if r[\'id\']==ref[\'id\'] else transform(catalog(small),ref_cat)\n            rec={\'path\':r[\'path\'],\'filter\':r.get(\'filter\'),\'status\':\'registered\',\'matrix_bin4\':matrix.tolist(),**stats}\n            grouped[r[\'filter\']].append((r,matrix))\n        except Exception as e:\n            rec={\'path\':r[\'path\'],\'status\':\'registration_rejected\',\'reason\':str(e)}\n        registration.append(rec);print(rec[\'status\'],r[\'path\'],flush=True)\n        atomic_json(out/\'registration.json\',registration)\n    masters={};integration={}\n    h,w=ref[\'shape\'];shape=(h//binning,w//binning)\n    for filt,items in grouped.items():\n        if len(items)<3:\n            integration[filt]={\'status\':\'insufficient_frames\',\'frames\':len(items)};continue\n        aligned=[];weights=[];seconds=0\n        for r,matrix in items:\n            if binning==4:a=cache(r)\n            else:\n                with fits.open(root/\'raw\'/r[\'path\'],memmap=False) as hdus:\n                    a=np.asarray(next(x.data for x in hdus if int(x.header.get(\'NAXIS\',0))==2),dtype=np.float32)\n                if binning!=1:a=bin_mean(a,binning)\n            # Only a scalar pedestal. Do not fit away extended nebulae automatically.\n            a=(a-float(r[\'background_adu\']))/float(r[\'exposure\'])\n            aligned.append(warp_affine(a,rebin_matrix(matrix,4,binning),shape,backend=backend))\n            weights.append((float(r[\'exposure\'])/max(float(r[\'noise_adu\']),1))**2)\n            seconds+=float(r[\'exposure\'])\n        weights=np.asarray(weights);weights=np.clip(weights/np.median(weights),0.25,4)\n        master=stack(aligned,StackConfig(method=\'sigma_clip_mean\',sigma=3,max_iterations=3,tile_size=384,backend=backend),weights=weights)\n        del aligned\n        masters[filt]=master\n        head=fits.Header();head[\'OBJECT\']=target;head[\'FILTER\']=filt;head[\'EXPTIME\']=seconds\n        head[\'NCOMBINE\']=len(items);head[\'SOFTBIN\']=binning;head[\'BUNIT\']=\'ADU/s\';head[\'CALSTAT\']=\'LIGHTS_ONLY\'\n        head[\'HISTORY\']=\'Scalar background removed; registered and sigma-clipped. No dark/flat correction.\'\n        fits.PrimaryHDU(master,header=head).writeto(out/(f\'master_{filt}_bin{binning}.fits\'),overwrite=True)\n        display_mono(master,(1800,1800)).save(out/(f\'master_{filt}_preview.png\'))\n        by_night={}\n        for r,_ in items:\n            entry=by_night.setdefault(r[\'night\'],{\'frames\':0,\'seconds\':0.})\n            entry[\'frames\']+=1;entry[\'seconds\']+=float(r[\'exposure\'])\n        integration[filt]={\'status\':\'integrated\',\'frames\':len(items),\'seconds\':seconds,\n                           \'by_night\':by_night,\'weights\':weights.tolist()}\n        print(\'MASTER\',filt,len(items),seconds,flush=True)\n    palette=None\n    if all(k in masters for k in [\'SII\',\'Ha\',\'OIII\']):channels=[\'SII\',\'Ha\',\'OIII\'];palette=\'SHO mapped colour\'\n    elif all(k in masters for k in [\'Ha\',\'OIII\']):channels=[\'Ha\',\'OIII\',\'OIII\'];palette=\'HOO mapped colour\'\n    elif all(k in masters for k in [\'R\',\'G\',\'B\']):channels=[\'R\',\'G\',\'B\'];palette=\'RGB approximate colour (not photometrically calibrated)\'\n    elif masters:\n        k=\'Ha\' if \'Ha\' in masters else next(iter(masters));channels=[k,k,k];palette=k+\' monochrome\'\n    else:raise ValueError(\'No channel has at least three registered frames\')\n    rgb=np.stack([masters[k] for k in channels],axis=-1)\n    valid=np.all(np.isfinite(rgb),axis=-1)\n    # Bounding crop, with invalid edge coverage retained in a separate mask.\n    yy,xx=np.where(valid)\n    if not len(yy):raise ValueError(\'No common registered coverage\')\n    y0,y1=int(yy.min()),int(yy.max()+1);x0,x1=int(xx.min()),int(xx.max()+1)\n    rgb=np.nan_to_num(rgb[y0:y1,x0:x1]);valid=valid[y0:y1,x0:x1]\n    gains=[]\n    for c in range(3):\n        vals=rgb[...,c][valid];span=max(float(np.percentile(vals,99.5)),1e-6)\n        gains.append(1/span)\n    # Global display gains, recorded explicitly; no local hue replacement.\n    rgb*=np.asarray(gains,dtype=np.float32)\n    rgb=gaussian_filter(rgb,sigma=(max(0.6,3/binning),max(0.6,3/binning),0))\n    lum=np.maximum(np.mean(rgb,axis=-1),1e-6)\n    target_lum=np.arcsinh(8*lum)/np.arcsinh(8)\n    rendered=rgb*(target_lum/lum)[...,None]\n    rendered=rendered/np.maximum(1,np.max(rendered,axis=-1,keepdims=True))\n    rendered=np.clip(rendered+0.008,0,1);rendered[~valid]=0\n    Image.fromarray(np.round(rendered*255).astype(\'uint8\')).save(out/\'colour_quicklook.png\')\n    tifffile.imwrite(out/\'colour_quicklook_16bit.tif\',np.round(rendered*65535).astype(\'uint16\'),photometric=\'rgb\')\n    fits.PrimaryHDU(valid.astype(\'uint8\')).writeto(out/\'common_coverage.fits\',overwrite=True)\n    report={\'target\':target,\'night\':night,\'binning\':binning,\'backend\':backend,\'palette\':palette,\n            \'reference\':ref[\'path\'],\'reference_selection\':\'explicit\' if reference_path else \'automatic_sharpness\',\n            \'observing_nights\':sorted({r[\'night\'] for items in grouped.values() if len(items)>=3 for r,_ in items}),\n            \'channels\':integration,\'display_gains\':gains,\'crop_xy\':[x0,y0,x1,y1],\n            \'limitations\':[\'Lights only: no matching darks/flats supplied.\',\'Quicklook uses global display balance, not measured filter transmission.\',\n                          \'Scalar sky offsets only; gradients/halos may remain.\',\'Candidate is a screening result, not guaranteed production suitability.\'],\n            \'selection\':{\'include_review\':include_review,\'overrides\':overrides},\'output\':str(out)}\n    atomic_json(out/\'processing_report.json\',report)\n    return report\n', 'finish.py': '"""Observation-derived presentation rendering, separate from archival integration.\n\nRestores the earlier B20 global-colour workflow: common sky anchors, star-excluded\ndiffuse scales, fixed spatial smoothing, and soft monotonic highlight curves.\nNo external image, generated detail, local hue replacement or star replacement.\nRun on Colab; linear masters are never altered.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport sep\nimport tifffile\nfrom astropy.io import fits\nfrom PIL import Image, ImageCms\nfrom scipy.ndimage import gaussian_filter\nfrom skimage.restoration import denoise_wavelet\n\nfrom archive import atomic_json\n\n\ndef mad_noise(a):\n    d=np.diff(a[::2],axis=1)\n    d=d[np.isfinite(d)]\n    return max(float(np.median(np.abs(d-np.median(d)))/.67448975/np.sqrt(2)),1e-9)\n\n\ndef common_crop(arrays):\n    valid=np.logical_and.reduce([np.isfinite(a) for a in arrays])\n    h,w=valid.shape\n    for margin in range(8,min(h,w)//4,2):\n        if valid[margin:-margin,margin:-margin].all():\n            return [margin,h-margin,margin,w-margin]\n    raise ValueError(\'No conservative rectangular common coverage\')\n\n\ndef star_mask(a):\n    a=np.ascontiguousarray(a,dtype=np.float32)\n    bg=sep.Background(a,bw=48,bh=48)\n    obj=sep.extract(a-bg.back(),4.5,err=bg.rms(),minarea=3)\n    obj=obj[(obj[\'a\']>.5)&(obj[\'a\']<8)&(obj[\'b\']/obj[\'a\']>.5)&(obj[\'npix\']<600)]\n    mask=np.zeros(a.shape,np.float32)\n    h,w=a.shape\n    for o in obj:\n        x,y=float(o[\'x\']),float(o[\'y\'])\n        r=float(np.clip(3.7*np.sqrt(o[\'a\']*o[\'b\']),3.25,12))\n        ir=int(np.ceil(r+2));ix,iy=int(x),int(y)\n        ya,yb=max(0,iy-ir),min(h,iy+ir+1);xa,xb=max(0,ix-ir),min(w,ix+ir+1)\n        yy,xx=np.mgrid[ya:yb,xa:xb]\n        m=np.clip((r+1-np.hypot(xx-x,yy-y))/2,0,1)\n        mask[ya:yb,xa:xb]=np.maximum(mask[ya:yb,xa:xb],m)\n    return mask,obj\n\n\ndef prepare(source, *, plane=False):\n    """Estimate only global display parameters on the archived bin4 masters."""\n    source=Path(source)\n    report=json.loads((source/\'processing_report.json\').read_text())\n    channels=[c for c,r in report[\'channels\'].items() if r[\'status\']==\'integrated\']\n    a={}\n    for c in channels:\n        path=source/f\'master_{c}_bin4.fits\'\n        if path.exists():a[c]=fits.getdata(path).astype(np.float32)\n        else:\n            b=int(report[\'binning\']);factor=4//b\n            v=fits.getdata(source/f\'master_{c}_bin{b}.fits\').astype(np.float32)\n            h,w=v.shape\n            a[c]=v[:h//factor*factor,:w//factor*factor].reshape(h//factor,factor,w//factor,factor).mean((1,3))\n    crop=common_crop(list(a.values()));y0,y1,x0,x1=crop\n    a={c:v[y0:y1,x0:x1] for c,v in a.items()}\n    ref=\'G\' if \'G\' in a else (\'Ha\' if \'Ha\' in a else channels[0])\n    mask,stars=star_mask(a[ref]);noise={c:mad_noise(v) for c,v in a.items()}\n    tiles=[];xy=[];h,w=a[ref].shape\n    for y in range(16,h-16,32):\n        for x in range(16,w-16,32):\n            tiles.append([np.median(a[c][y-12:y+12,x-12:x+12]) for c in channels])\n            xy.append([1,x/w-.5,y/h-.5])\n    tiles=np.asarray(tiles);xy=np.asarray(xy)\n    rank=lambda v:np.argsort(np.argsort(v))/len(v)\n    sky_channels=[c for c in [\'Ha\',\'OIII\'] if c in a] or [ref]\n    score=np.mean([rank(tiles[:,channels.index(c)]) for c in sky_channels],axis=0)\n    anchors=score<np.percentile(score,12)\n    backgrounds={c:float(np.median(tiles[anchors,i])-.5*noise[c]) for i,c in enumerate(channels)}\n    slopes={c:[0.,0.] for c in channels}\n    if plane:\n        # Slopes only: common low-emission tiles, constrained and auditable.\n        chosen=(score<np.percentile(score,35)) & (np.max(np.abs(xy[:,1:]),axis=1)>.25)\n        enough=all(np.count_nonzero(chosen&(xy[:,1]*sx>0)&(xy[:,2]*sy>0))>=4 for sx in [-1,1] for sy in [-1,1])\n        if enough:\n            for i,c in enumerate(channels):\n                use=chosen.copy()\n                for _ in range(4):\n                    coef=np.linalg.lstsq(xy[use],tiles[use,i],rcond=None)[0]\n                    resid=tiles[:,i]-xy@coef\n                    sig=max(float(np.median(np.abs(resid[use]-np.median(resid[use])))*1.4826),noise[c]*.1)\n                    use=chosen&(np.abs(resid-np.median(resid[use]))<3*sig)\n                limit=max(float(np.ptp(np.percentile(tiles[chosen,i],[10,90]))),noise[c])\n                scale=min(1.,limit/max(abs(coef[1])+abs(coef[2]),1e-12))\n                slopes[c]=(coef[1:]*scale).tolist()\n                adjusted=tiles[anchors,i]-xy[anchors,1:]@np.asarray(slopes[c])\n                backgrounds[c]=float(np.median(adjusted)-.5*noise[c])\n    keep=1-mask;den=gaussian_filter(keep,4)\n    scales={}\n    for c in channels:\n        v=a[c]-backgrounds[c]\n        if plane:\n            yy,xx=np.ogrid[:h,:w]\n            v=v-slopes[c][0]*((xx+.5)/w-.5)-slopes[c][1]*((yy+.5)/h-.5)\n        interp=gaussian_filter(v*keep,4)/np.maximum(den,.001)\n        diffuse=v*keep+interp*mask\n        scales[c]=max(float(np.percentile(gaussian_filter(diffuse,1),99.3)),8*noise[c])\n    gains={};phot=[]\n    if {\'R\',\'G\',\'B\'}<=a.keys():\n        yy,xx=np.mgrid[-12:13,-12:13];rr=np.hypot(xx,yy);disk=rr<4.5;ann=(rr>8)&(rr<12)\n        for o in stars:\n            x,y=int(o[\'x\']),int(o[\'y\'])\n            if not(14<x<w-14 and 14<y<h-14):continue\n            flux=[];peak=[]\n            for c in \'RGB\':\n                p=a[c][y-12:y+13,x-12:x+13];sky=np.median(p[ann])\n                flux.append(float(np.sum(p[disk]-sky)));peak.append(float(p[disk].max()-sky))\n            if all(f>40*noise[c] and pk>8*noise[c] for c,f,pk in zip(\'RGB\',flux,peak)) and max(peak)<25:\n                phot.append(flux)\n        if len(phot)<20:raise ValueError(\'Too few unsaturated reference stars for empirical RGB balance\')\n        phot=np.asarray(phot)\n        gains={\'R\':float(np.median(phot[:,1]/phot[:,0])),\'G\':1.,\'B\':float(np.median(phot[:,1]/phot[:,2]))}\n        if not all(.2<v<5 for v in gains.values()):raise ValueError(\'Implausible stellar colour gains\')\n    return {\'channels\':channels,\'crop_bin4\':crop,\'backgrounds\':backgrounds,\'slopes\':slopes,\n            \'scales\':scales,\'noise_bin4\':noise,\'rgb_gains\':gains,\'rgb_balance_stars\':len(phot),\n            \'star_count\':len(stars),\'sky_anchor_tiles\':int(anchors.sum()),\'plane\':plane,\n            \'star_mask_use\':\'Only for global diffuse-scale estimation; no star replacement\'}\n\n\ndef tone(a, strength=6., shoulder=2.):\n    v=np.arcsinh(strength*np.maximum(a,0))\n    return .012+.975*v/(v+shoulder)\n\n\ndef compress_gamut(rgb, gray):\n    """Reduce chroma only as needed to retain hue at the display boundaries."""\n    gray=np.clip(gray,.012,.99)\n    d=rgb-gray[...,None]\n    lo=np.maximum(-d.min(-1),1e-9);hi=np.maximum(d.max(-1),1e-9)\n    scale=np.minimum(1,np.minimum((gray-.012)/lo,(.99-gray)/hi))\n    return np.clip(gray[...,None]+d*scale[...,None],.012,.99)\n\n\ndef presentation_settings(palette, overrides=None):\n    defaults={\'denoise\':.88,\'smooth_native\':3.2,\'saturation\':1.55,\'power\':1.08,\n              \'oxygen\':.7,\'local_contrast\':.22,\'strength\':8,\'shoulder\':1.9}\n    if palette==\'RGB\':\n        defaults.update(saturation=1.15,strength=6,shoulder=2,local_contrast=.15)\n    elif palette.endswith(\'_Monochrome\'):\n        defaults.update(saturation=1,local_contrast=.24,strength=8,shoulder=1.8)\n    return {**defaults,**(overrides or {})}\n\n\ndef render(report, *, settings=None, output_name=\'finished\', parameters=None):\n    overrides=dict(settings or {})\n    settings=presentation_settings(\'SHO\',overrides)\n    source=Path(report.get(\'source_run\',report[\'output\']))\n    inp=Path(report[\'output\']);binning=int(report[\'binning\'])\n    p=parameters or prepare(source,plane=settings.get(\'plane\',False))\n    out=inp/output_name;out.mkdir(exist_ok=True)\n    crop=[int(v*4/binning) for v in p[\'crop_bin4\']];y0,y1,x0,x1=crop\n    ch={};metrics={}\n    for c in p[\'channels\']:\n        a=fits.getdata(inp/f\'master_{c}_bin{binning}.fits\').astype(np.float32)[y0:y1,x0:x1]\n        if not np.isfinite(a).all():raise ValueError(\'Invalid pixels inside presentation crop\')\n        a-=p[\'backgrounds\'][c]\n        h,w=a.shape;yy,xx=np.ogrid[:h,:w]\n        a-=p[\'slopes\'][c][0]*((xx+.5)/w-.5)+p[\'slopes\'][c][1]*((yy+.5)/h-.5)\n        halo=settings.get(\'halo_models\',{}).get(c)\n        if halo:\n            # Optional measured smooth halo, with no inward extrapolation into\n            # an unsampled stellar core. Model and strength remain in the report.\n            scale=4/binning;cx,cy=halo[\'center_bin4\']\n            rr=np.hypot((xx+.5)/scale-.5-cx,(yy+.5)/scale-.5-cy)\n            rr=np.maximum(rr,halo.get(\'minimum_radius_bin4\',0))\n            model=halo[\'amplitude\']/(1+(rr/halo[\'radius_bin4\'])**2)**halo[\'beta\']\n            a-=float(settings.get(\'halo_strength\',.85))*model\n        noise=mad_noise(a)\n        den=denoise_wavelet(a,sigma=noise,method=\'BayesShrink\',mode=\'soft\',wavelet=\'sym4\',wavelet_levels=3,rescale_sigma=False,channel_axis=None)\n        mix=float(settings.get(\'denoise\',.8));a=(1-mix)*a+mix*den\n        sigma=float(settings.get(\'smooth_native\',2.6))/binning\n        ch[c]=gaussian_filter(a,sigma).astype(np.float32)\n        metrics[c]={\'noise_before\':noise,\'noise_after\':mad_noise(ch[c]),\'smooth_sigma_pixels\':sigma}\n    palettes={};sc=p[\'scales\']\n    if {\'Ha\',\'OIII\'}<=ch.keys():\n        H=np.maximum(ch[\'Ha\']/sc[\'Ha\'],0);O=np.maximum(ch[\'OIII\']/sc[\'OIII\'],0)*float(settings.get(\'oxygen\',.75))\n        palettes[\'HOO\']=np.stack([.85*H,.08*H+.4*O,.55*O],-1)\n        # Explicit two-filter alternative, not a claimed SHO measurement.\n        palettes[\'HOO_Amber_Blue\']=np.stack([.85*H,.39*H+.24*O,.04*H+.72*O],-1)\n    if {\'SII\',\'Ha\',\'OIII\'}<=ch.keys():\n        S=np.maximum(ch[\'SII\']/sc[\'SII\'],0)\n        palettes[\'SHO\']=np.stack([.35*S+.35*H,.28*H+.10*S+.20*O,.70*O],-1)\n    if {\'R\',\'G\',\'B\'}<=ch.keys():\n        rgb=np.stack([ch[c]*p[\'rgb_gains\'][c] for c in \'RGB\'],-1)\n        rgbscale=float(settings.get(\'rgb_scale\',min(sc[c]*p[\'rgb_gains\'][c] for c in \'RGB\')))\n        palettes[\'RGB\']=rgb/rgbscale\n    mono=\'Ha\' if \'Ha\' in ch else p[\'channels\'][0]\n    if not palettes:palettes[mono+\'_Monochrome\']=np.repeat(np.maximum(ch[mono]/sc[mono],0)[...,None],3,-1)\n    icc=ImageCms.ImageCmsProfile(ImageCms.createProfile(\'sRGB\')).tobytes()\n    exports={};palette_settings={}\n    for name,rgb in palettes.items():\n        display=presentation_settings(name,overrides)\n        palette_settings[name]=display\n        img=tone(rgb,strength=float(display[\'strength\']),shoulder=float(display[\'shoulder\']))\n        # A common display curve and saturation adjustment; no spatial hue map.\n        img=.012+.975*np.power(np.clip((img-.012)/.975,0,1),float(display[\'power\']))\n        lum=img@np.array([.2126,.7152,.0722],np.float32)\n        saturation=float(display[\'saturation\'])\n        img=lum[...,None]+saturation*(img-lum[...,None])\n        amount=float(display[\'local_contrast\'])\n        detail=gaussian_filter(lum,2/binning)-gaussian_filter(lum,40/binning)\n        ratio=np.clip((lum+amount*detail)/np.maximum(lum,.012),.8,1.2)\n        img*=ratio[...,None]\n        img=compress_gamut(img,lum*ratio)\n        im=Image.fromarray(np.uint8(np.rint(img*255)))\n        im.save(out/(name+\'.png\'),icc_profile=icc)\n        t=np.uint16(np.rint(img*65535));tifffile.imwrite(out/(name+\'_16bit.tif\'),t,photometric=\'rgb\',extratags=[(34675,\'B\',len(icc),icc,False)])\n        assert np.array_equal(t,tifffile.imread(out/(name+\'_16bit.tif\')))\n        im.thumbnail((2400,1800),Image.Resampling.LANCZOS);im.save(out/(name+\'_preview.jpg\'),quality=96,icc_profile=icc)\n        exports[name]={\'shape\':list(img.shape),\'min\':float(img.min()),\'max\':float(img.max()),\n                       \'black_fraction\':float((img.max(-1)==0).mean()),\'clipped_fraction\':float((img.max(-1)>=1).mean())}\n    result={\'target\':report[\'target\'],\'input_run\':str(inp),\'source_run\':str(source),\'binning\':binning,\n            \'parameters\':p,\'settings\':settings,\'palette_settings\':palette_settings,\'metrics\':metrics,\'exports\':exports,\'output\':str(out),\n            \'limitations\':[\'Lights only; not dark/flat calibrated.\',\'Global mapped narrowband colours; RGB uses empirical stellar balance, not catalogue calibration.\',\n                          \'Monotonic per-channel display curve; presentation pixels are not photometric fluxes.\'],\n            \'method\':\'B20 global matrix and soft highlight curve; wavelet noise reduction; no external or generated detail.\'}\n    atomic_json(out/\'render_report.json\',result)\n    print(\'FINISHED\',report[\'target\'],out,exports,flush=True)\n    return result\n', 'report.py': '"""Summarize measured usability; lunar context is descriptive, not a veto."""\nfrom collections import Counter, defaultdict\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom astropy.coordinates import get_body_barycentric\nfrom astropy.time import Time\n\nfrom archive import atomic_json\n\n\ndef summarize(root):\n    root=Path(root)\n    rows=json.loads((root/\'quality/frame_report.json\').read_text())\n    overrides=json.loads((root/\'quality/overrides.json\').read_text()) if (root/\'quality/overrides.json\').exists() else {}\n    grouped=defaultdict(list)\n    for r in rows:\n        r=dict(r)\n        if r[\'path\'] in overrides:\n            r[\'status\']=overrides[r[\'path\']][\'status\']\n            r[\'reasons\']+=[\'Manual review: \'+overrides[r[\'path\']].get(\'reason\',\'\')]\n        grouped[(r[\'night\'],r.get(\'object\') or \'Unknown\',r.get(\'filter\') or \'Unknown\')].append(r)\n    summary=[];moon={}\n    for (night,obj,filt),group in sorted(grouped.items()):\n        counts=Counter(r[\'status\'] for r in group)\n        for r in group:\n            if night not in moon and r.get(\'date_obs\'):\n                try:\n                    t=Time(r[\'date_obs\'],scale=\'utc\')\n                    e=get_body_barycentric(\'earth\',t).xyz.value\n                    s=get_body_barycentric(\'sun\',t).xyz.value\n                    m=get_body_barycentric(\'moon\',t).xyz.value\n                    cosine=float(np.dot(e-m,s-m)/(np.linalg.norm(e-m)*np.linalg.norm(s-m)))\n                    moon[night]=round(100*(1+cosine)/2,1)\n                except (ValueError,TypeError):pass\n        keep=[r for r in group if r[\'status\'] in {\'candidate\',\'accepted\'}]\n        summary.append({\'night\':night,\'object\':obj,\'filter\':filt,\'total\':len(group),\n                        \'candidate\':len(keep),\'review\':counts[\'review\'],\'reject\':counts[\'reject\'],\n                        \'planetary_or_test\':counts[\'planetary_or_test\'],\'non_image\':counts[\'non_image\'],\n                        \'candidate_minutes\':round(sum(float(r.get(\'exposure\') or 0) for r in keep)/60,1),\n                        \'moon_illumination_percent_approx\':moon.get(night)})\n    atomic_json(root/\'quality/summary.json\',summary)\n    text=[\'# OMI archive quality report\',\'\',\n          \'Candidate = passes conservative screening; final use still requires successful registration and visual review. \'\n          \'Review frames are excluded by default. Rejected originals remain archived. Planetary/test frames need their own workflow.\',\'\',\n          \'Moon phase is approximate context from FITS DATE-OBS and Astropy built-in ephemerides. It is not an exclusion rule or proof of the cause of a defect.\',\'\',\n          \'| Night | Target | Filter | Total | Candidate | Review | Reject | Planet/test | Non-image | Candidate min | Moon lit % |\',\n          \'|---|---|---|---:|---:|---:|---:|---:|---:|---:|---:|\']\n    for r in summary:\n        text.append(\'| \'+\' | \'.join(str(r[k]) for k in [\'night\',\'object\',\'filter\',\'total\',\'candidate\',\'review\',\'reject\',\'planetary_or_test\',\'non_image\',\'candidate_minutes\',\'moon_illumination_percent_approx\'])+\' |\')\n    text+=[\'\',\'## Processing limits\',\'\',\n           \'- Supplied lights are not dark/flat calibrated unless matching calibration is added separately.\',\n           \'- Sky background, gradients, stellar elongation, banding and saturation are measured separately.\',\n           \'- Five native-pixel star samples are supplemented by full-field morphology to catch long trails and arcs.\',\n           \'- Automatic colour outputs are review images. SHO/HOO colours are mappings; RGB is not photometrically calibrated.\',\n           \'- The notebook produces 4×4-binned review masters by default; binning=1 produces native-resolution masters.\',\n           \'- No outside image pixels or generated astronomical detail are used.\']\n    (root/\'QUALITY_REPORT.md\').write_text(\'\\n\'.join(text)+\'\\n\')\n    return summary\n', 'test_workflow.py': "import io\nimport json\nimport zipfile\n\nimport numpy as np\nimport pytest\nfrom astropy.io import fits\n\nimport archive\nfrom quality import star_measurements, refine_field_metrics, inspect_one\n\n\ndef test_reference_override_can_anchor_inspected_field():\n    from process import choose_reference\n    wrong={'path':'wrong-field-ha.fits','filter':'Ha','fwhm_px':2.0}\n    correct={'path':'correct-field-oiii.fits','filter':'OIII','fwhm_px':3.0}\n    assert choose_reference([wrong,correct],correct['path']) is correct\n    assert choose_reference([wrong,correct]) is wrong\n\n\ndef test_reference_override_cannot_bypass_frame_selection():\n    from process import choose_reference\n    selected=[{'path':'accepted.fits','filter':'Ha','fwhm_px':3.0}]\n    with pytest.raises(ValueError,match='selected light frame'):\n        choose_reference(selected,'rejected.fits')\n\n\n@pytest.mark.parametrize('name',['../outside.fits','/absolute.fits','a/../../x','a\\\\..\\\\x'])\ndef test_unsafe_paths(name):\n    with pytest.raises(ValueError):archive.safe_relative(name)\n\n\ndef test_download_query_preserves_shared_key():\n    u=archive.download_url('https://www.dropbox.com/scl/fo/example?rlkey=private&dl=0')\n    assert 'rlkey=private' in u and 'dl=1' in u\n    with pytest.raises(ValueError):archive.download_url('https://dropbox.com.evil.example/a')\n\n\nclass Response:\n    def __init__(self,payload,content_type='application/binary'):\n        self.payload=payload\n        self.headers={'Content-Length':str(len(payload)),'Content-Type':content_type}\n    def __enter__(self):return self\n    def __exit__(self,*a):pass\n    def raise_for_status(self):pass\n    def iter_content(self,n):yield self.payload\n\n\ndef payload():\n    b=io.BytesIO();fits.PrimaryHDU(np.arange(256,dtype='uint16').reshape(16,16)).writeto(b)\n    return b.getvalue()\n\n\ndef test_archive_resume_and_missing_receipt(tmp_path,monkeypatch):\n    item={'path':'night/target/a.fits','url':'https://www.dropbox.com/scl/fi/example?dl=0'}\n    monkeypatch.setattr(archive.requests,'get',lambda *a,**k:Response(payload()))\n    a=archive.import_one(item,tmp_path,retries=1)\n    assert a['status']=='downloaded' and a['fits_error'] is None\n    assert archive.import_one(item,tmp_path,retries=1)['status']=='verified_existing'\n    receipt=tmp_path/'receipts/night/target/a.fits.json';receipt.unlink()\n    assert archive.import_one(item,tmp_path,retries=1)['sha256']==a['sha256']\n    assert json.loads(receipt.read_text())['bytes']==len(payload())\n\n\ndef test_bad_sources_archived_but_html_rejected(tmp_path,monkeypatch):\n    item={'path':'night/broken.fits','url':'https://www.dropbox.com/scl/fi/example'}\n    monkeypatch.setattr(archive.requests,'get',lambda *a,**k:Response(b'broken-source'))\n    r=archive.import_one(item,tmp_path,retries=1)\n    assert r['status']=='downloaded' and r['fits_error']\n    assert (tmp_path/'raw/night/broken.fits').read_bytes()==b'broken-source'\n    item['path']='night/login.fits'\n    monkeypatch.setattr(archive.requests,'get',lambda *a,**k:Response(b'<html>login</html>','text/html'))\n    assert archive.import_one(item,tmp_path,retries=1)['status']=='failed'\n    assert not (tmp_path/'raw/night/login.fits').exists()\n\n\ndef test_conflicting_original_never_overwritten(tmp_path,monkeypatch):\n    p=tmp_path/'raw/night/a.fits';p.parent.mkdir(parents=True);p.write_bytes(b'original')\n    monkeypatch.setattr(archive.requests,'get',lambda *a,**k:Response(payload()))\n    r=archive.import_one({'path':'night/a.fits','url':'https://www.dropbox.com/a'},tmp_path,retries=1)\n    assert r['status']=='failed' and p.read_bytes()==b'original'\n\n\ndef test_zip_traversal(tmp_path,monkeypatch):\n    b=io.BytesIO()\n    with zipfile.ZipFile(b,'w') as z:z.writestr('../escape.fits',payload())\n    monkeypatch.setattr(archive.requests,'get',lambda *a,**k:Response(b.getvalue()))\n    with pytest.raises(ValueError):\n        archive.import_folder_link('https://www.dropbox.com/folder',tmp_path/'drive','night',tmp_path/'stage')\n    assert not (tmp_path/'escape.fits').exists()\n\n\ndef test_trails_are_measured_and_hot_pixels_excluded():\n    rng=np.random.default_rng(12);yy,xx=np.mgrid[:1024,:1024]\n    def field(sx,sy):\n        a=rng.normal(500,4,(1024,1024)).astype('float32')\n        for y in range(60,1000,85):\n            for x in range(60,1000,85):\n                a+=800*np.exp(-0.5*((xx-x)**2/sx**2+(yy-y)**2/sy**2))\n        for y,x in rng.integers(0,1024,(120,2)):a[y,x]+=18000\n        return a\n    good,_=star_measurements(field(2.5,2.5))\n    bad,_=star_measurements(field(11,2))\n    assert good['sample_stars']>30 and good['axis_ratio']>0.8 and good['fwhm_px']>4\n    assert bad['sample_stars']>30 and bad['axis_ratio']<0.4 and bad['trail_fraction']>0.8\n\n\ndef test_full_field_catches_trails_with_sparse_native_samples():\n    yy,xx=np.mgrid[:512,:512];rng=np.random.default_rng(9)\n    a=rng.normal(500,1,(512,512)).astype('float32')\n    for y in range(25,500,50):\n        for x in range(25,500,50):a+=80*np.exp(-.5*((xx-x)**2/7**2+(yy-y)**2/1.1**2))\n    r=refine_field_metrics({'status':'review','reasons':[]},a)\n    assert r['field_stars']>30 and r['status']=='reject'\n\n\ndef test_broad_sensor_step_flagged_even_with_round_stars():\n    yy,xx=np.mgrid[:512,:512];rng=np.random.default_rng(19)\n    a=rng.normal(500,1,(512,512)).astype('float32')\n    for y in range(25,500,50):\n        for x in range(25,500,50):a+=80*np.exp(-.5*((xx-x)**2+(yy-y)**2)/1.4**2)\n    r=refine_field_metrics({'status':'candidate','reasons':[],'row_jump_adu':100,'noise_adu':5},a)\n    assert r['status']=='review' and 'abrupt full-width row discontinuity' in r['reasons']\n\n\ndef test_metadata_only_sync_file_is_not_a_failed_exposure(tmp_path):\n    root=tmp_path/'archive';(root/'raw').mkdir(parents=True)\n    path=root/'raw/sync.fits';fits.PrimaryHDU().writeto(path)\n    r=inspect_one(path,root)\n    assert r['status']=='non_image' and r['night']=='root'\n\n\ndef test_repeated_folder_import_refreshes_updated_night(tmp_path,monkeypatch):\n    def zipped(names):\n        b=io.BytesIO()\n        with zipfile.ZipFile(b,'w') as z:\n            for name in names:z.writestr(name,payload())\n        return b.getvalue()\n    incoming=iter([zipped(['a.fits']),zipped(['a.fits','b.fits'])])\n    monkeypatch.setattr(archive.requests,'get',lambda *a,**k:Response(next(incoming)))\n    url='https://www.dropbox.com/folder'\n    archive.import_folder_link(url,tmp_path/'drive','night',tmp_path/'stage',keep_staging=True)\n    records=archive.import_folder_link(url,tmp_path/'drive','night',tmp_path/'stage')\n    assert len(records)==2 and (tmp_path/'drive/raw/night/b.fits').exists()\n    assert not list((tmp_path/'stage').glob('*.zip'))\n\n\ndef test_finishing_curve_preserves_order_without_hard_highlight_clipping():\n    from finish import tone\n    x=np.logspace(-6,6,1000)\n    y=tone(x)\n    assert np.all(np.diff(y)>0) and y.min()>.011 and y.max()<1\n\n\ndef test_gamut_compression_preserves_chroma_direction():\n    from finish import compress_gamut\n    rgb=np.array([[1.4,.3,-.2],[.1,.2,.3]],dtype=np.float32)\n    gray=np.array([.4,.2],dtype=np.float32)\n    out=compress_gamut(rgb,gray)\n    assert out.min()>=.01199 and out.max()<=.99001\n    np.testing.assert_allclose(out[1],rgb[1],atol=1e-7)\n    before=rgb[0]-gray[0];after=out[0]-gray[0]\n    np.testing.assert_allclose(after/np.linalg.norm(after),before/np.linalg.norm(before),atol=1e-6)\n\n\ndef test_common_crop_excludes_missing_coverage():\n    from finish import common_crop\n    a=np.ones((120,160),np.float32);a[:10]=np.nan;a[:,:12]=np.nan\n    crop=common_crop([a,np.ones_like(a)])\n    y0,y1,x0,x1=crop\n    assert np.isfinite(a[y0:y1,x0:x1]).all()\n\n\n@pytest.mark.parametrize('overrides',[None,{'saturation':1.3,'strength':7}])\ndef test_mixed_filters_do_not_change_individual_palette_exports(tmp_path,overrides):\n    from finish import render\n    import tifffile\n    yy,xx=np.mgrid[:64,:80]\n    channels=['SII','Ha','OIII','R','G','B']\n    rng=np.random.default_rng(42)\n    for i,c in enumerate(channels):\n        a=(.05+(.3+.07*i)*np.exp(-((xx-30-i)**2+(yy-32)**2)/500)\n           +rng.normal(0,.002,xx.shape)).astype('float32')\n        fits.PrimaryHDU(a).writeto(tmp_path/f'master_{c}_bin4.fits')\n    def run(chosen,label):\n        report={'target':'test','binning':4,'output':str(tmp_path),\n                'channels':{c:{'status':'integrated'} for c in chosen}}\n        p={'channels':chosen,'crop_bin4':[8,56,8,72],\n           'backgrounds':{c:0 for c in chosen},'slopes':{c:[0,0] for c in chosen},\n           'scales':{c:1 for c in chosen},'rgb_gains':{c:1 for c in 'RGB'}}\n        return render(report,parameters=p,settings=overrides,output_name=label)\n    mixed=run(channels,'mixed')\n    run(channels[:3],'narrowband');run(channels[3:],'broadband')\n    for palette,group in [('SHO','narrowband'),('HOO','narrowband'),\n                          ('HOO_Amber_Blue','narrowband'),('RGB','broadband')]:\n        np.testing.assert_array_equal(tifffile.imread(tmp_path/'mixed'/f'{palette}_16bit.tif'),\n                                      tifffile.imread(tmp_path/group/f'{palette}_16bit.tif'))\n    if overrides:\n        for s in mixed['palette_settings'].values():\n            assert s['saturation']==overrides['saturation'] and s['strength']==overrides['strength']\n"}
for name,source in MODULES.items():
    (WORK/name).write_text(source)
saved=ROOT/'workflow';saved.mkdir(exist_ok=True)
for name,source in MODULES.items():(saved/name).write_text(source)
(saved/'source_hashes.json').write_text(json.dumps({n:hashlib.sha256(s.encode()).hexdigest() for n,s in MODULES.items()},indent=2))
(saved/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
checks=subprocess.run([sys.executable,'-m','pytest','-q',str(WORK/'test_workflow.py')],cwd=WORK,capture_output=True,text=True)
print(checks.stdout)
assert checks.returncode==0, checks.stderr
import importlib
importlib.invalidate_caches()
for filename in MODULES:
    module=filename.removesuffix('.py')
    if module in sys.modules:importlib.reload(sys.modules[module])


### Transfer and archive

Original paths, bytes and SHA-256 checksums are retained. A repeat run verifies existing files and retries incomplete ones. Conflicting originals are not overwritten. Damaged source FITS are preserved and reported. A future folder import downloads a ZIP to temporary Colab disk; use individual observing-night links so it fits available space.

In [ ]:
#@title 4. Import originals directly into Drive
from archive import import_manifest, import_folder_link
if IMPORT_MODE=='New night folder':
    assert NEW_NIGHT_LINK and NEW_NIGHT_LABEL, 'Enter the Dropbox night-folder link and date label.'
    transfers=import_folder_link(NEW_NIGHT_LINK,ROOT,NEW_NIGHT_LABEL)
elif IMPORT_MODE=='Saved manifest':
    manifest=ROOT/'source_manifest.complete.private.json'
    if not manifest.exists():manifest=ROOT/'source_manifest.private.json'
    assert manifest.exists(), 'For a new archive, choose New night folder.'
    transfers=import_manifest(json.loads(manifest.read_text()),ROOT,workers=4)
    failures=[r for r in transfers if r['status']=='failed']
    assert not failures, f'{len(failures)} transfers need a retry; see transfer_report.json'
else:
    print('Using archived FITS; transfer skipped.')
print('Originals:',ROOT/'raw')


### Quality screening

The report distinguishes candidate, review and rejected frames. The Moon's illumination is context, not a rejection criterion. Measurements include native star profiles, full-field trails, noise, background, gradients, saturation and row/column patterns. Review and rejected frames stay on Drive and are excluded from automatic processing.

The supplied data contain lights only. Without matching darks/flats, fixed-pattern noise and illumination errors may remain. Inspect `QUALITY_REPORT.md`, `quality/frame_report.csv`, and the per-frame JPG review sheets. Manual decisions can be recorded in `quality/overrides.json` by relative raw path, with `status` and `reason`.

In [ ]:
#@title 5. Screen the archive and summarize observing sessions
from quality import scan
from report import summarize
rows=scan(ROOT,workers=4)
summary=summarize(ROOT)
for r in summary:
    print(f"{r['night']} | {r['object']} | {r['filter']}: {r['candidate']}/{r['total']} candidates, "
          f"{r['review']} review, {r['reject']} rejected, {r['planetary_or_test']} planetary/test")
print('Report:',ROOT/'QUALITY_REPORT.md')


### Integrate and render

The default **2×2 binning** produces presentation masters. Use **1** for native resolution or **4** for faster review. Registration must pass star-match, residual and scale checks. Different camera/gain/geometry groups are not silently mixed. At least three registered frames per channel are required. Sharp stars alone do not confirm pointing: after inspecting a mismatched field, set **REFERENCE_PATH** to the accepted exposure's relative raw path. Leave it blank for automatic selection. A blank observing night combines compatible archived sessions; actual contributions per night are recorded for each filter.

The screening quicklook is saved separately. Finishing restores the earlier B20 method: common sky anchors, star-excluded diffuse scales, wavelet noise reduction, global colour matrices, soft highlights and modest luminance contrast. Linear masters remain unchanged.

HOO and its amber/blue alternative both use only Hα and OIII; SHO requires measured SII as well. A single filter stays monochrome. RGB uses empirical unsaturated-star balance, not catalogue photometric calibration. Global display choices are recorded in `finished/render_report.json`. No reference-photo pixels, generated structure, local hue patches or replacement stars are added.

In [ ]:
#@title 6. Process the chosen target on the GPU and save results
from process import process_target
if RUN_PROCESSING:
    result=process_target(ROOT,TARGET,night=OBSERVING_NIGHT or None,binning=int(BINNING),backend='gpu',reference_path=REFERENCE_PATH or None)
    print(json.dumps(result,indent=2))
    print('Results:',result['output'])
    preview=Path(result['output'])/'colour_quicklook.png'
    if FINISH_IMAGES:
        from finish import render
        presentation=render(result)
        print('Finished presentations:',presentation['output'])
        available=presentation['exports']
        preferred=next((p for p in ['SHO','HOO','RGB'] if p in available),next(iter(available)))
        preview=Path(presentation['output'])/(preferred+'.png')
    if DISPLAY_IMAGES:
        from IPython.display import display,Image
        display(Image(filename=str(preview),width=1000))
else:
    print('Archive and quality report complete. Processing disabled in configuration.')


Saved layout:

```
OMI_Archive/
  raw/                       original Dropbox paths and bytes
  receipts/                  per-file SHA-256 and transfer receipts
  quality/                   measurements, decisions, caches and review sheets
  processed/<target>/<run>/  linear FITS masters, PNG, 16-bit TIFF, coverage, provenance
    finished/                presentation PNG/TIFF, previews and rendering parameters
  workflow/                  exact scripts and environment used
  QUALITY_REPORT.md
```

Keep the shared-link manifests private. Colab is not an always-on scheduler. Background jobs stop if its runtime is reclaimed; completed originals and reports remain in Drive.

[Dropbox folder downloads](https://help.dropbox.com/sync/download-entire-folders) · [Colab runtime limits](https://research.google.com/colaboratory/faq.html)